# 남해안 조위관측소 기상(풍향/풍속) 데이터 수집 및 전처리

- **API**: 국립해양조사원 surveyWind (GetSurveyWindApiService)
- **수집 방식**: min=60 파라미터로 1시간 간격 직접 수신
- **출력 파일**:
  - `national_weather_master_1hr.csv` : 전국 26개 관측소
  - `namhae_weather_master_1hr.csv` : 남해안 15개 관측소
- **컬럼**: datetime / obs_name / lat / lon / wind_dir / wind_speed / anomaly_flag

---

### 실행 순서
1. CELL 1~5 순서대로 실행 (함수 정의)
2. CELL 6 실행 (수집 시작)
   - 429 또는 중단 시 다시 실행하면 완료된 관측소 자동 스킵
3. CELL 7 실행 (전처리 & 저장)
4. CELL 8 실행 (검증 & 다운로드)

In [ ]:
# CELL 1 - 라이브러리
# !pip install scipy -q

import requests
import pandas as pd
import numpy as np
import time
import os
from datetime import datetime, timedelta
from scipy.spatial import cKDTree
from google.colab import files

In [ ]:
# CELL 2 - 설정값

API_KEY  = "b90866d8453868988d1975f2a31fef831560bfef923d8a8937b67aabd7c3cca6"
BASE_URL = "https://apis.data.go.kr/1192136/surveyWind/GetSurveyWindApiService"

START_DATE       = "20250101"
END_DATE         = datetime.today().strftime("%Y%m%d")
REQUEST_DELAY    = 0.1   # API 요청 간격 (초)
MAX_INTERP_HOURS = 3     # 선형보간 허용 최대 연속 결측 시간
BACKUP_FILE      = "_raw_backup.csv"

date_list = pd.date_range(START_DATE, END_DATE, freq="D").strftime("%Y%m%d").tolist()
print(f"수집 기간: {START_DATE} ~ {END_DATE} ({len(date_list)}일)")
print(f"예상 호출 수: {len(date_list) * 26:,}번 (일일 한도 10,000건)")

In [ ]:
# CELL 3 - 관측소 목록

NAMHAE_STATIONS = [
    {"obs_code": "DT_0005", "obs_name": "부산",      "lat": 35.0983, "lon": 129.0350},
    {"obs_code": "DT_0056", "obs_name": "부산항신항", "lat": 35.0700, "lon": 128.7700},
    {"obs_code": "DT_0063", "obs_name": "가덕도",    "lat": 35.0167, "lon": 128.8167},
    {"obs_code": "DT_0062", "obs_name": "마산",      "lat": 35.1833, "lon": 128.5667},
    {"obs_code": "DT_0014", "obs_name": "통영",      "lat": 34.8333, "lon": 128.4333},
    {"obs_code": "DT_0061", "obs_name": "삼천포",    "lat": 34.9167, "lon": 128.0667},
    {"obs_code": "DT_0029", "obs_name": "거제도",    "lat": 34.8000, "lon": 128.6333},
    {"obs_code": "DT_0026", "obs_name": "고흥발포",  "lat": 34.6167, "lon": 127.2833},
    {"obs_code": "DT_0016", "obs_name": "여수",      "lat": 34.7467, "lon": 127.7650},
    {"obs_code": "DT_0049", "obs_name": "광양",      "lat": 34.9000, "lon": 127.7167},
    {"obs_code": "DT_0031", "obs_name": "거문도",    "lat": 34.0283, "lon": 127.3067},
    {"obs_code": "DT_0027", "obs_name": "완도",      "lat": 34.3967, "lon": 126.7583},
    {"obs_code": "DT_0028", "obs_name": "진도",      "lat": 34.4833, "lon": 126.3167},
    {"obs_code": "DT_0007", "obs_name": "목포",      "lat": 34.7783, "lon": 126.3767},
    {"obs_code": "DT_0094", "obs_name": "서거차도",  "lat": 34.1833, "lon": 125.9500},
]

EXTRA_STATIONS = [
    {"obs_code": "DT_0003", "obs_name": "영광",   "lat": 35.2667, "lon": 126.4833},
    {"obs_code": "DT_0018", "obs_name": "군산",   "lat": 35.9750, "lon": 126.5617},
    {"obs_code": "DT_0025", "obs_name": "보령",   "lat": 36.4167, "lon": 126.4833},
    {"obs_code": "DT_0067", "obs_name": "안흥",   "lat": 36.6783, "lon": 126.1300},
    {"obs_code": "DT_0001", "obs_name": "인천",   "lat": 37.4519, "lon": 126.5922},
    {"obs_code": "DT_0091", "obs_name": "포항",   "lat": 36.0500, "lon": 129.3667},
    {"obs_code": "DT_0020", "obs_name": "울산",   "lat": 35.4833, "lon": 129.3833},
    {"obs_code": "DT_0006", "obs_name": "묵호",   "lat": 37.5500, "lon": 129.1167},
    {"obs_code": "DT_0012", "obs_name": "속초",   "lat": 38.2067, "lon": 128.5933},
    {"obs_code": "DT_0004", "obs_name": "제주",   "lat": 33.5279, "lon": 126.5433},
    {"obs_code": "DT_0010", "obs_name": "서귀포", "lat": 33.2440, "lon": 126.5610},
]

for s in NAMHAE_STATIONS: s["region"] = "남해안"
for s in EXTRA_STATIONS:   s["region"] = "기타"

ALL_STATIONS = NAMHAE_STATIONS + EXTRA_STATIONS
NAMHAE_CODES = {s["obs_code"] for s in NAMHAE_STATIONS}
print(f"전체 관측소: {len(ALL_STATIONS)}개 | 남해안: {len(NAMHAE_CODES)}개")

In [ ]:
# CELL 4 - 수집 함수

def fetch_one_day(obs_code: str, req_date: str, retry: int = 3) -> list:
    """
    하루치 수집 (min=60, 1시간 간격 24건).
    - 429 발생 시 자동 대기 후 재시도
    - resultCode 03 (NODATA): 빈 리스트 반환
    """
    params = {
        "serviceKey": API_KEY,
        "type":       "json",
        "obsCode":    obs_code,
        "reqDate":    req_date,
        "min":        60,
        "numOfRows":  300,
        "pageNo":     1,
    }

    for attempt in range(retry):
        try:
            resp = requests.get(BASE_URL, params=params, timeout=15)

            if resp.status_code == 429:
                wait = 5 * (attempt + 1)
                print(f"  [429] {obs_code} {req_date} -> {wait}s 대기 후 재시도")
                time.sleep(wait)
                continue

            if resp.status_code != 200 or not resp.text.strip():
                return []

            data = resp.json()
            rc   = data.get("header", {}).get("resultCode", "")

            if rc == "03":
                return []
            if rc != "00":
                print(f"  [WARN] rc={rc} {obs_code} {req_date}")
                return []

            items = data["body"]["items"]["item"]
            return [items] if isinstance(items, dict) else items

        except Exception as e:
            print(f"  [ERROR] {obs_code} {req_date}: {e}")
            time.sleep(2)
    return []


def get_done_codes() -> set:
    """백업 파일에서 이미 완료된 관측소 코드를 반환합니다."""
    if not os.path.exists(BACKUP_FILE):
        return set()
    try:
        df     = pd.read_csv(BACKUP_FILE, usecols=["obs_code"])
        counts = df["obs_code"].value_counts()
        done   = {code for code, cnt in counts.items() if cnt > len(date_list) * 2}
        return done
    except:
        return set()


def collect_all(stations: list) -> pd.DataFrame:
    """
    전체 관측소 x 전체 기간 순차 수집.
    - 관측소 완료마다 백업 파일에 append 저장
    - 재시작 시 완료된 관측소 자동 스킵
    """
    done_codes = get_done_codes()
    todo       = [s for s in stations if s["obs_code"] not in done_codes]
    start_time = time.time()

    if done_codes:
        print(f"이전 진행 감지 - 완료된 관측소 {len(done_codes)}개 스킵, 남은 관측소 {len(todo)}개")
    else:
        print("수집 시작")

    for stn in todo:
        code, name = stn["obs_code"], stn["obs_name"]
        records    = []

        for i, d in enumerate(date_list):
            rows = fetch_one_day(code, d)
            for r in rows:
                records.append({
                    "obs_code":   code,
                    "obs_name":   name,
                    "lat":        stn["lat"],
                    "lon":        stn["lon"],
                    "region":     stn["region"],
                    "raw_time":   r.get("obsrvnDt"),
                    "wind_dir":   r.get("wndrct"),
                    "wind_speed": r.get("wspd"),
                })

            time.sleep(REQUEST_DELAY)

            if (i + 1) % 30 == 0:
                elapsed = (time.time() - start_time) / 60
                print(f"  [{name}] {d} ({i+1}/{len(date_list)}일) | 경과 {elapsed:.1f}분", end="\r")

        elapsed = (time.time() - start_time) / 60
        if records:
            df_stn = pd.DataFrame(records)
            if os.path.exists(BACKUP_FILE):
                df_stn.to_csv(BACKUP_FILE, mode="a", header=False, index=False, encoding="utf-8-sig")
            else:
                df_stn.to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")
            print(f"  [{name}] 완료 {len(records):,}행 저장 | 경과 {elapsed:.1f}분")
        else:
            print(f"  [{name}] 수집 0행 - obsCode 확인 필요")

    if os.path.exists(BACKUP_FILE):
        raw_df = pd.read_csv(BACKUP_FILE, encoding="utf-8-sig")
        print(f"전체 수집 완료: {len(raw_df):,}행 | 총 {(time.time()-start_time)/60:.1f}분")
        return raw_df
    else:
        print("수집된 데이터 없음")
        return pd.DataFrame(
            columns=["obs_code","obs_name","lat","lon","region","raw_time","wind_dir","wind_speed"]
        )

In [ ]:
# CELL 5 - 전처리 함수

def circular_mean(angles):
    """
    풍향 원형 평균.
    일반 평균은 350도+10도=180도 오류 발생 -> sin/cos 이용해 올바르게 계산.
    """
    a = angles.dropna()
    if a.empty: return np.nan
    r = np.deg2rad(a)
    return float(np.degrees(np.arctan2(np.sin(r).mean(), np.cos(r).mean())) % 360)


def resample_1hr(g: pd.DataFrame) -> pd.DataFrame:
    """1시간 리샘플 (풍향: 원형평균, 풍속: 산술평균)."""
    g = g.set_index("datetime").sort_index()
    res = pd.DataFrame()
    res["wind_speed"] = g["wind_speed"].resample("1H").mean()
    res["wind_dir"]   = g["wind_dir"].resample("1H").apply(circular_mean)
    for c in ["obs_code","obs_name","lat","lon","region"]:
        res[c] = g[c].iloc[0]
    return res.reset_index()


def preprocess(raw: pd.DataFrame, stations: list) -> pd.DataFrame:
    """
    전처리 파이프라인:
      1. datetime 파싱 & 수치형 변환
      2. 이상치 마스킹 (풍속 >60, 풍향 >360 -> NaN)
      3. 1시간 리샘플
      4. 전체 날짜 x 관측소 완전 격자 생성
      5. 선형보간 (연속 결측 MAX_INTERP_HOURS 이하)
      6. 이웃 관측소 보간 (긴 결측)
      7. anomaly_flag 부여
    """
    print("전처리 시작")
    if len(raw) == 0:
        print("수집된 데이터 없음")
        return pd.DataFrame()

    # 1) 파싱
    raw["datetime"]   = pd.to_datetime(raw["raw_time"], errors="coerce")
    raw["wind_dir"]   = pd.to_numeric(raw["wind_dir"],   errors="coerce")
    raw["wind_speed"] = pd.to_numeric(raw["wind_speed"], errors="coerce")
    raw.dropna(subset=["datetime"], inplace=True)
    print(f"  [1] 파싱: {len(raw):,}행")

    # 2) 이상치 마스킹
    raw.loc[~raw["wind_speed"].between(0, 60),  "wind_speed"] = np.nan
    raw.loc[~raw["wind_dir"].between(0, 360),   "wind_dir"]   = np.nan
    print(f"  [2] 이상치 마스킹 완료")

    # 3) 1시간 리샘플
    df = pd.concat(
        [resample_1hr(g) for _, g in raw.groupby("obs_code")],
        ignore_index=True
    )
    print(f"  [3] 1H 리샘플: {len(df):,}행")

    # 4) 완전 격자 생성
    full_range = pd.date_range(
        pd.to_datetime(START_DATE),
        pd.to_datetime(END_DATE) + timedelta(hours=23),
        freq="1H"
    )
    grid = pd.DataFrame(
        [(dt, s["obs_code"]) for dt in full_range for s in stations],
        columns=["datetime","obs_code"]
    )
    grid = grid.merge(pd.DataFrame(stations), on="obs_code", how="left")
    df   = grid.merge(
        df[["datetime","obs_code","wind_dir","wind_speed"]],
        on=["datetime","obs_code"], how="left"
    )
    print(f"  [4] 격자: {len(df):,}행")

    # 5) 선형보간
    df = df.sort_values(["obs_code","datetime"])
    for col in ["wind_dir","wind_speed"]:
        df[col] = df.groupby("obs_code")[col].transform(
            lambda s: s.interpolate(method="linear", limit=MAX_INTERP_HOURS, limit_direction="both")
        )
    print(f"  [5] 선형보간 완료 (최대 {MAX_INTERP_HOURS}시간)")

    # 6) 이웃 관측소 보간
    df["anomaly_flag"] = ""
    filled = []
    for dt, grp in df.groupby("datetime"):
        nan_mask = grp[["wind_dir","wind_speed"]].isna().any(axis=1)
        if not nan_mask.any(): continue
        tree = cKDTree(grp[["lat","lon"]].values)
        for li in grp[nan_mask].index:
            row = df.loc[li]
            _, idxs = tree.query([row["lat"],row["lon"]], k=len(grp))
            for col in ["wind_dir","wind_speed"]:
                if pd.notna(df.at[li,col]): continue
                for ki in idxs:
                    ni = grp.index[ki]
                    if ni == li: continue
                    v = df.at[ni,col]
                    if pd.notna(v):
                        df.at[li,col] = v
                        if li not in filled: filled.append(li)
                        break
    for idx in filled:
        df.at[idx,"anomaly_flag"] = "인접보간됨"
    print(f"  [6] 이웃 보간: {len(filled)}셀")

    # 7) anomaly_flag 부여
    flags = []
    for _, row in df.iterrows():
        if row["anomaly_flag"] == "인접보간됨":
            flags.append("인접보간됨"); continue
        flags.append(
            "정상" if pd.notna(row["wind_speed"]) and pd.notna(row["wind_dir"]) else "결측"
        )
    df["anomaly_flag"] = flags

    # 8) 컬럼 정리
    df["datetime"] = df["datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")
    df = df[[
        "datetime","obs_name","lat","lon",
        "wind_dir","wind_speed","anomaly_flag","region"
    ]].sort_values(["datetime","obs_name"]).reset_index(drop=True)

    print(f"전처리 완료: {len(df):,}행")
    return df

In [ ]:
# CELL 6 - 수집 실행
# 429로 중단돼도 괜찮습니다.
# 다시 실행하면 완료된 관측소는 자동으로 스킵됩니다.

raw_df = collect_all(ALL_STATIONS)

In [ ]:
# CELL 7 - 전처리 & 저장
# 26개 관측소 전부 완료된 후 실행하세요.

if os.path.exists(BACKUP_FILE):
    check     = pd.read_csv(BACKUP_FILE, usecols=["obs_code","obs_name"])
    done      = check.groupby(["obs_code","obs_name"]).size().reset_index(name="rows")
    remaining = [s["obs_name"] for s in ALL_STATIONS if s["obs_code"] not in done["obs_code"].values]
    print(f"수집된 관측소: {len(done)}개 / {len(ALL_STATIONS)}개")
    if remaining:
        print(f"남은 관측소: {remaining}")
        print("-> CELL 6을 다시 실행하세요.")
    else:
        print("전체 수집 완료 - 전처리 진행")

final_df = preprocess(raw_df, ALL_STATIONS)

if len(final_df) > 0:
    # 전국
    national_df = final_df.drop(columns=["region"])
    national_df.to_csv("national_weather_master_1hr.csv", index=False, encoding="utf-8-sig")
    print(f"[전국] national_weather_master_1hr.csv ({len(national_df):,}행)")

    # 남해안
    namhae_df = final_df[final_df["region"] == "남해안"].drop(columns=["region"])
    namhae_df.to_csv("namhae_weather_master_1hr.csv", index=False, encoding="utf-8-sig")
    print(f"[남해안] namhae_weather_master_1hr.csv ({len(namhae_df):,}행)")

In [ ]:
# CELL 8 - 검증 및 다운로드

if len(final_df) > 0:
    print("--- 검증 (남해안) ---")
    print(f"기간: {namhae_df['datetime'].min()} ~ {namhae_df['datetime'].max()}")
    print(f"관측소 수: {namhae_df['obs_name'].nunique()}개")
    print(f"관측소 목록: {sorted(namhae_df['obs_name'].unique().tolist())}")

    print("\n결측률:")
    for col in ["wind_dir","wind_speed"]:
        rate = namhae_df[col].isna().mean()
        print(f"  {col}: {rate:.2%}")

    print("\nanomaly_flag 분포:")
    print(namhae_df["anomaly_flag"].value_counts().to_string())

    print("\n샘플 (head 5):")
    print(namhae_df.head(5).to_string(index=False))

    files.download("namhae_weather_master_1hr.csv")
    files.download("national_weather_master_1hr.csv")
    print("다운로드 완료")